# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates Idempotent UPSERT/DELETE logic on a simulated stateful Iceberg table using MERGE INTO.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day07_Incremental_Merge") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()


In [ ]:
# Setup Target Table
spark.sql("CREATE DATABASE IF NOT EXISTS local.ehdip_silver_db")
spark.sql("DROP TABLE IF EXISTS local.ehdip_silver_db.patients")
spark.sql("""
CREATE TABLE local.ehdip_silver_db.patients (
    payload_id STRING,
    name STRING,
    status STRING,
    updated_at TIMESTAMP
)
USING iceberg
""")

# Initial Insert (Create)
spark.sql("""
INSERT INTO local.ehdip_silver_db.patients
VALUES ('pat_1', 'John Doe', 'active', timestamp '2023-10-01 10:00:00')
""")
print("Initial State:")
spark.sql("SELECT * FROM local.ehdip_silver_db.patients").show()


In [ ]:
# Simulate CDC Updates
updates_data = [
    # Update existing
    ('pat_1', 'John Doe', 'inactive', '2023-10-02 10:00:00', 'u'),
    # Create new
    ('pat_2', 'Jane Smith', 'active', '2023-10-02 10:05:00', 'c'),
    # Delete (simulated by a later event, but let's just insert one to delete for demo if we had it)
    # For this demo, let's say pat_3 was there and now deleted.
]
updates_df = spark.createDataFrame(updates_data, ["payload_id", "name", "status", "updated_at", "op"])
# Cast string to timestamp
from pyspark.sql.functions import to_timestamp
updates_df = updates_df.withColumn("updated_at", to_timestamp("updated_at"))

updates_df.createOrReplaceTempView("updates")

print("CDC Updates:")
spark.sql("SELECT * FROM updates").show()


In [ ]:
# Perform Merge
merge_query = """
MERGE INTO local.ehdip_silver_db.patients t
USING updates u
ON t.payload_id = u.payload_id
WHEN MATCHED AND u.op = 'd' THEN DELETE
WHEN MATCHED AND u.op = 'u' AND u.updated_at > t.updated_at THEN UPDATE SET t.name = u.name, t.status = u.status, t.updated_at = u.updated_at
WHEN NOT MATCHED AND u.op IN ('c', 'u') THEN INSERT (payload_id, name, status, updated_at) VALUES (u.payload_id, u.name, u.status, u.updated_at)
"""
spark.sql(merge_query)

print("State after Merge:")
spark.sql("SELECT * FROM local.ehdip_silver_db.patients").show()
